In [74]:
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path

In [75]:
import ee, geemap

# Initialize earthengine-api connection
try:
    ee.Initialize(project='edadson-2025')
    print("Initialized.")
except Exception:
    ee.Authenticate()  # follow the link; choose/allow the project when prompted
    ee.Initialize(project='edadson-2025')
    print("Initialized.")

Initialized.


In [ ]:
# Site table: the same location2 as the R pipeline (covariates/cew_sites.csv,
# written by cew_covariates_hpc.R and checked by step1/step1_data.R):
# unique (longitude, latitude) of the Lawton et al. counts with longitude >= -105,
# sorted by longitude then latitude, numbered 1, 2, ...
repo = Path.home() / "Desktop/cew_seasonal_bayes"          # project folder on the Mac
data_root = Path("/Volumes/cmjone25/Data/Original/pest-occurrence/corn_earworm")

sites = pd.read_csv(repo / "covariates/cew_sites.csv")

# Rebuild it from the raw counts with the same rule and check that they agree
# (catches a changed data file or a stale cew_sites.csv)
lawton = pd.read_csv(data_root / "doi_10_5061_dryad_m0cfxpp5x__v20221206/Lawton_etal_data.csv")
rebuilt = (lawton.loc[lawton["longitude"] >= -105, ["longitude", "latitude"]]
           .drop_duplicates().sort_values(["longitude", "latitude"]).reset_index(drop=True))
rebuilt["location2"] = np.arange(1, len(rebuilt) + 1)
assert len(rebuilt) == len(sites), f"{len(rebuilt)} sites in the counts vs {len(sites)} in cew_sites.csv"
assert (rebuilt["location2"].values == sites["location2"].values).all()
assert np.allclose(rebuilt[["longitude", "latitude"]].values, sites[["longitude", "latitude"]].values)

unique_points = sites[["location2", "longitude", "latitude"]]
print(len(unique_points), "sites; location2 matches covariates/cew_sites.csv")

In [ ]:
# One point per site, carrying location2 and its coordinates (exported with the
# EVI so step1_data.R can check the join)
target_proj = ee.Projection('EPSG:4326')
features = [
    ee.Feature(
        ee.Geometry.Point([p["longitude"], p["latitude"]], proj=target_proj),
        {"location2": int(p["location2"]), "longitude": float(p["longitude"]), "latitude": float(p["latitude"])}
    )
    for _, p in unique_points.iterrows()
]
fc = ee.FeatureCollection(features)

In [ ]:
start_date = "2008-01-01"
end_date   = "2020-12-31"

# --- 3. Load reflectance + QA collections -----------------------------------
mcd43a4 = ee.ImageCollection("MODIS/061/MCD43A4").filterDate(start_date, end_date)
mcd43a2 = ee.ImageCollection("MODIS/061/MCD43A2").filterDate(start_date, end_date)

join_filter = ee.Filter.equals(leftField="system:time_start", rightField="system:time_start")
joined = ee.Join.inner().apply(mcd43a4, mcd43a2, join_filter)

# --- 4. Compute EVI, mask by QA, and sample at points — all in one pass ----
def process_pair(pair):
    pair = ee.Feature(pair)
    refl = ee.Image(pair.get("primary"))
    qa   = ee.Image(pair.get("secondary"))

    red  = refl.select("Nadir_Reflectance_Band1").multiply(0.0001)
    nir  = refl.select("Nadir_Reflectance_Band2").multiply(0.0001)
    blue = refl.select("Nadir_Reflectance_Band3").multiply(0.0001)

    evi = red.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": nir, "RED": red, "BLUE": blue}
    ).rename("EVI")

    # 061 quality bands: 0=best, 1=good, 2-3=magnitude inversion, 4=fill
    q1 = qa.select("BRDF_Albedo_Band_Quality_Band1")
    q2 = qa.select("BRDF_Albedo_Band_Quality_Band2")
    q3 = qa.select("BRDF_Albedo_Band_Quality_Band3")
    good_quality = q1.lte(1).And(q2.lte(1)).And(q3.lte(1))

    evi_masked = evi.updateMask(good_quality)

    date = refl.date().format("YYYY-MM-dd")
    native_proj = evi_masked.projection()

    sampled = evi_masked.reduceRegions(
        collection=fc,
        reducer=ee.Reducer.first(),
        scale=500,
        crs=native_proj
    )

    # .get() tolerates points with no valid-quality pixel (returns null, not an error)
    sampled = sampled.map(
        lambda f: f.set("EVI", f.get("first")).set("date", date)
    )
    return sampled

extracted = ee.FeatureCollection(joined.map(process_pair)).flatten()

# --- 5. Export ---------------------------------------------------------------
# Keyed by location2 (+ coordinates for the check in step1_data.R). Download
# from Drive and save as
#   <data_root>/MCD43_EVI_points_location2.csv
# step1/step1_data.R reads that file when it exists (the older
# MCD43_EVI_points.csv, keyed by the full-timeseries ids, otherwise).
task = ee.batch.Export.table.toDrive(
    collection=extracted,
    description="MCD43_EVI_points_location2",
    folder="EarthEngineExports",
    fileNamePrefix="MCD43_EVI_points_location2",
    fileFormat="CSV",
    selectors=["location2", "longitude", "latitude", "date", "EVI"],
)
task.start()
print("Export task started:", task.id)